<a href="https://colab.research.google.com/github/shyoonCS/DataAnalysis/blob/main/%EB%A8%B8%EC%8B%A0%EB%9F%AC%EB%8B%9D_%EA%B3%BC%EC%A0%9C(%EC%BD%94%EB%93%9C%2C_%EA%B3%BC%EC%A0%9C%EC%88%98%ED%96%89%EB%B3%B4%EA%B3%A0%EC%84%9C)_202534_364010(%EC%9C%A4%EC%84%A0%ED%9D%AC).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###머신러닝 출석과제 [K-근접이웃 분류기 구현하기]
* 학  번 : 202534-364010
* 성  명 : 윤선희
---



##<span style="color:blue"> **1. 코드, 코드설명(주석)**</span>

In [114]:
#수행에 필요한 라이브러리 불러오기
import csv      # csv 파일 읽기 위한 라이브러리
import random   # 데이터 랜덤 처리 라이브러리 : 난수 생성, 리스트의 데이터를 랜덤으로 순서 변경 또는 데이터 추출
import math     # 수학함수 제공
import pandas as pd # DataFrame, Series 데이터 객체 사용

(1) 데이터 불러오기

In [115]:
#CSV 파일을 읽어서 리스트 형태로 저장, 데이터 구성 : # float형 특성값 4개, 정수형 레이블 1개(class)
iris_data = []
with open("iris_KNN.csv", "r", encoding="utf-8") as f:
    fdata = csv.reader(f)
    for row in fdata:
        features = list(map(float, row[:4])) # map함수를 이용하여 list 4개 값을 float 변환하여 featuers list 구성
        label = int(row[4]) # class label
        iris_data.append((features, label))  #리스트 형태로 iris_data 구성 ex: ([5.0, 3.5, 1.6, 0.6], 1)

(2) 학습데이터, 테스트데이터 분할

In [116]:
 # 2.학습 데이터 / 테스트 데이터 분할
random.shuffle(iris_data)    # 리스트의 데이터를 랜덤으로 순서 변경
train_data = iris_data[:100] #0~99까지 100개의 데이터를 뽑아 학습데이터로 할당
test_data = iris_data[100:]  #100~ 나머지 데이터를 테스트데이터로 할당

(3) 거리 계산 함수(유클리디안 거리)
* **유클리디안 거리(Euclidean distance)** : 두 점 사이의 직선 거리를 계산하는 방식


In [117]:
# 3. 거리 계산 함수 (유클리디안 거리)
def euclidean_distance(x1, x2): # x1 : train_data의 feature, x2 :test_data의feture
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(x1, x2))) # zip : x1, x2 리스트의 각 요소를 튜플로 처리하여, 차이 제곱근 계산

In [118]:
 # 4. KNN 분류기 함수 (직접 구현)

"""
k-최근접 이웃(KNN) 알고리즘으로 test_sample의 레이블을 예측하는 함수
train: 학습 데이터 리스트 [(특징벡터, 레이블), ...]
test_sample: 예측하고자 하는 데이터의 특징벡터
k: 참조할 최근접 이웃의 개수
"""
def knn_predict(train, test_sample, k):
    # 1. 학습 데이터 각각과 테스트 샘플 사이의 거리를 계산하고 저장
    distances = []  # (거리, 레이블)을 저장할 빈 리스트
    for features, label in train:  # 학습 데이터 하나씩 반복
        dist = euclidean_distance(features, test_sample)  # 유클리드 거리 계산
        distances.append((dist, label))  # 거리와 해당 레이블을 튜플로 저장

    # 2. 거리를 기준으로 오름차순 정렬 (가장 가까운 데이터가 앞으로)
    distances.sort(key=lambda x: x[0])

    # 3. 가장 가까운 k개의 데이터 선택
    k_nearest = distances[:k]  # 상위 k개 요소 추출

    # 4. k개의 데이터 중 어떤 레이블이 가장 많은지 세기
    label_count = {}  # 레이블별 등장 횟수를 저장할 딕셔너리
    for dist, label in k_nearest:
        if label in label_count:
            label_count[label] += 1  # 이미 있으면 1 증가
        else:
            label_count[label] = 1  # 없으면 1로 초기화

    # 5. 가장 많이 등장한 레이블 반환 (다수결)
    predicted_label = max(label_count, key=label_count.get)  # value 기준 최대값
    return predicted_label




In [119]:
# 5. 여러 K값에 대해 분류 수행 및 정확도 계산
# -------------------------------------------------------
results = []
for k in [5, 10, 20, 30]:
    correct = 0
    for features, label in test_data:
        pred = knn_predict(train_data, features, k)
        if pred == label:
            correct += 1
    accuracy = correct / len(test_data)
    results.append((k, accuracy))


In [120]:
# 6. 결과를 표 형태로 출력
# -------------------------------------------------------
df = pd.DataFrame(results, columns=["K 값", "정확도"])
print(df.to_string(index=False))

 K 값  정확도
   5 0.98
  10 0.96
  20 0.94
  30 0.94


1.3. 코드 1-1[7] 수정
* 퍼셉트론 객체 생성 및 학습(학습 반복횟수 : 500회,학습률(Learning Rate) : 0.1-> 0.5로 수정하여 빠르게 학습)
* 활성함수 : **계단함수**(입력값이 0 이상이면 1, 0 미만이면 0을 출력하는 함수), 출력을 0과 1로 **이진분류(classification)**